# 이전 결과가 좋아서 안함!

# YOLOv8n 약한 튜닝 학습
- **파일명**: `12_train_yolov8n_light_tuning.ipynb`
- **저장 경로**: `N:\개인\이수빈\3.13_Mini_Project\scripts\`
- **목적**: scale 중심 보수적 튜닝 (가까운 화재 탐지 개선)
- **작성일**: 2026-03-04
- **예상 소요**: 약 2~2.5시간

### 이전 튜닝 실패 원인
- 6개 파라미터를 한꺼번에 크게 변경 → 원본 특성 손상

### 이번 전략
- **scale만 핵심 변경** (0.5 → 0.7): 가까운 화재 탐지 개선
- **degrees 소폭 추가** (0.0 → 5.0): 기울어진 화재 대응
- **나머지 전부 Default 유지**: 검증된 설정을 건드리지 않음


In [ ]:
# =============================================================================
# [셀 1] 라이브러리 임포트
# =============================================================================

# ultralytics: YOLO 모델 학습/추론 라이브러리
from ultralytics import YOLO

# pathlib: 폴더/파일 경로를 다루는 라이브러리 (os.path보다 안정적)
from pathlib import Path

# datetime: 학습 시작/종료 시간을 기록하기 위한 라이브러리
from datetime import datetime


In [ ]:
# =============================================================================
# [셀 2] 프로젝트 경로 설정
# =============================================================================

# 프로젝트 최상위 폴더 경로 (본인 PC에 맞게 수정)
PROJECT_ROOT = Path(r"N:\개인\이수빈\3.13_Mini_Project")

# 데이터셋 설정 파일 경로 (train/val 이미지 위치가 적혀있는 yaml 파일)
# 이 파일이 YOLO에게 "학습 데이터는 여기, 검증 데이터는 여기" 알려줌
DATA_YAML = str(PROJECT_ROOT / "DATASET" / "5000_split_70_15" / "data.yaml")

# 학습 결과가 저장될 폴더 이름
# 이전 튜닝: yolov8n_tuned → 이번: yolov8n_light_tuned (구분하기 위해)
RESULT_NAME = "yolov8n_light_tuned"

# 결과 저장 상위 폴더
RESULT_DIR = str(PROJECT_ROOT / "results")

# 경로 확인 (실수 방지)
print(f"data.yaml 존재: {Path(DATA_YAML).exists()}")
print(f"결과 저장 위치: {Path(RESULT_DIR) / RESULT_NAME}")


In [ ]:
# =============================================================================
# [셀 3] TDD - 학습 전 검증 테스트
# =============================================================================

def test_data_yaml_exists(data_yaml_path):
    """테스트 1: data.yaml 파일이 존재하는지 확인"""
    # data.yaml이 없으면 YOLO가 데이터를 찾을 수 없음
    assert Path(data_yaml_path).exists(), f"data.yaml 없음: {data_yaml_path}"
    print("✅ 테스트 1 통과: data.yaml 존재 확인")

def test_base_model_loadable():
    """테스트 2: YOLOv8n 기본 모델이 정상 로드되는지 확인"""
    # yolov8n.pt: YOLO가 제공하는 사전학습된 기본 모델 (처음부터 학습하는 게 아님)
    model = YOLO("yolov8n.pt")
    assert model is not None, "모델 로드 실패!"
    print("✅ 테스트 2 통과: YOLOv8n 모델 정상 로드")
    return model

def test_hyperparams_reasonable(params):
    """테스트 3: 하이퍼파라미터가 합리적 범위인지 확인"""
    # scale은 0~1 사이여야 함 (0=변형없음, 1=최대변형)
    assert 0 <= params["scale"] <= 1, f"scale 범위 초과: {params['scale']}"
    # degrees는 0~45 사이가 적절 (45도 이상 회전은 비현실적)
    assert 0 <= params["degrees"] <= 45, f"degrees 범위 초과: {params['degrees']}"
    # patience는 5~30 사이가 적절
    assert 5 <= params["patience"] <= 30, f"patience 범위 초과: {params['patience']}"
    # epochs는 50~300 사이가 적절
    assert 50 <= params["epochs"] <= 300, f"epochs 범위 초과: {params['epochs']}"
    print("✅ 테스트 3 통과: 하이퍼파라미터 범위 정상")


In [ ]:
# =============================================================================
# [셀 4] 하이퍼파라미터 설정 — 이번 튜닝의 핵심
# =============================================================================

# 이번에 변경할 파라미터와 이유를 정리
params = {
    # ====== 핵심 변경 ======
    
    # scale: 이미지 확대/축소 비율
    # Default=0.5 → 0.7로 변경
    # 왜? 0.7이면 이미지를 30%~170% 크기로 변형하며 학습
    # 효과: 화재가 크게 보이는 경우(=가까운 화재)를 더 많이 연습
    # 주의: 이전 실패(0.9)보다 훨씬 보수적
    "scale": 0.7,
    
    # degrees: 이미지 회전 각도
    # Default=0.0 → 5.0으로 변경
    # 왜? 실제 CCTV는 약간 기울어져 있을 수 있음
    # 효과: 살짝 기울어진 화재도 탐지할 수 있게 됨
    # 주의: 이전 실패(15.0)보다 훨씬 보수적
    "degrees": 5.0,
    
    # ====== Default 유지 (건드리지 않음) ======
    
    # translate: 이미지 이동 비율 (Default 0.1 유지)
    "translate": 0.1,
    
    # hsv_h: 색상 변형 (Default 0.015 유지)
    "hsv_h": 0.015,
    
    # hsv_s: 채도 변형 (Default 0.7 유지)
    "hsv_s": 0.7,
    
    # hsv_v: 밝기 변형 (Default 0.4 유지)
    "hsv_v": 0.4,
    
    # ====== 이전 실패에서 배운 것: 이것들은 끔 ======
    
    # copy_paste: 객체 복사-붙여넣기 (끔)
    # 이전에 0.3으로 켰다가 성능 하락 → 소규모 데이터에선 해로움
    "copy_paste": 0.0,
    
    # mixup: 이미지 2장을 겹치는 기법 (끔)
    # 이전에 0.15로 켰다가 성능 하락 → 화재 특성을 흐리게 만듦
    "mixup": 0.0,
    
    # flipud: 상하 반전 (끔)
    # 화재는 위로 올라가는 특성 → 뒤집으면 비현실적
    "flipud": 0.0,
    
    # ====== 학습 설정 ======
    
    # epochs: 최대 학습 반복 횟수
    "epochs": 100,
    
    # batch: 한 번에 몇 장씩 학습할지 (GPU 메모리에 맞춤)
    "batch": 16,
    
    # patience: 성능이 안 오르면 몇 에폭 후 학습 중단할지
    # 이전에 50이었는데 → 15로 수정 (불필요한 학습 방지)
    "patience": 15,
}

# 변경된 것만 한눈에 보기
print("=" * 60)
print("📋 이번 튜닝 변경 사항 요약")
print("=" * 60)
print(f"  scale:      0.5(Default) → {params['scale']}  ⬆️ 핵심 변경")
print(f"  degrees:    0.0(Default) → {params['degrees']}  ⬆️ 소폭 변경")
print(f"  patience:   50(이전) → {params['patience']}  ✅ 수정")
print(f"  copy_paste: 0.3(이전) → {params['copy_paste']}  ✅ 끔")
print(f"  mixup:      0.15(이전) → {params['mixup']}  ✅ 끔")
print(f"  flipud:     0.1(이전) → {params['flipud']}  ✅ 끔")
print(f"  나머지:     Default 유지")
print("=" * 60)


In [ ]:
# =============================================================================
# [셀 5] TDD 테스트 실행
# =============================================================================

# 테스트 1: data.yaml 존재 확인
test_data_yaml_exists(DATA_YAML)

# 테스트 2: 모델 로드 확인
model = test_base_model_loadable()

# 테스트 3: 파라미터 범위 확인
test_hyperparams_reasonable(params)

print("\n모든 사전 테스트 통과! 학습을 시작할 수 있습니다.")


In [ ]:
# =============================================================================
# [셀 6] 학습 실행 — ⚠️ 약 2~2.5시간 소요
# =============================================================================

# 학습 시작 시간 기록
start_time = datetime.now()
print(f"\n🚀 학습 시작: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"예상 소요 시간: 약 2~2.5시간 (RTX 4060 Ti 기준)")
print("=" * 60)

# YOLO 학습 실행
# model.train()에 위에서 설정한 파라미터들을 전달
results = model.train(
    data=DATA_YAML,            # 데이터 설정 파일 경로
    epochs=params["epochs"],   # 최대 100번 반복 학습
    batch=params["batch"],     # 한 번에 16장씩 학습
    imgsz=640,                 # 입력 이미지 크기 640x640 (YOLO 기본값)
    patience=params["patience"],  # 15 에폭 동안 성능 안 오르면 조기 종료
    device=0,                  # GPU 0번 사용 (RTX 4060 Ti)
    project=RESULT_DIR,        # 결과 저장 상위 폴더
    name=RESULT_NAME,          # 결과 저장 하위 폴더명
    exist_ok=True,             # 같은 이름 폴더 있어도 덮어쓰기 허용

    # === Augmentation 파라미터 ===
    scale=params["scale"],         # 0.7: 핵심 변경 (가까운 화재 대응)
    degrees=params["degrees"],     # 5.0: 소폭 변경 (기울기 대응)
    translate=params["translate"], # 0.1: Default 유지
    hsv_h=params["hsv_h"],         # 0.015: Default 유지
    hsv_s=params["hsv_s"],         # 0.7: Default 유지
    hsv_v=params["hsv_v"],         # 0.4: Default 유지
    copy_paste=params["copy_paste"],  # 0.0: 끔
    mixup=params["mixup"],            # 0.0: 끔
    flipud=params["flipud"],          # 0.0: 끔
)

# 학습 종료 시간 기록
end_time = datetime.now()
# 소요 시간 계산 (초 → 분으로 변환)
elapsed_minutes = (end_time - start_time).total_seconds() / 60

print("\n" + "=" * 60)
print(f"✅ 학습 완료!")
print(f"소요 시간: {elapsed_minutes:.1f}분 ({elapsed_minutes/60:.1f}시간)")
print(f"종료 시간: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)


In [ ]:
# =============================================================================
# [셀 7] 학습 결과 확인 (Validation 성능)
# =============================================================================

# 학습된 모델의 가중치 파일 경로
best_model_path = Path(RESULT_DIR) / RESULT_NAME / "weights" / "best.pt"
last_model_path = Path(RESULT_DIR) / RESULT_NAME / "weights" / "last.pt"

print(f"\n📁 저장된 모델 파일:")
print(f"  best.pt: {best_model_path}")
print(f"  (best = 학습 중 Validation 성능이 가장 좋았던 시점의 모델)")
print(f"  last.pt: {last_model_path}")
print(f"  (last = 마지막 에폭의 모델)")

# 모델 파일이 생성되었는지 확인
print(f"\n  best.pt 존재: {best_model_path.exists()}")
print(f"  last.pt 존재: {last_model_path.exists()}")


In [ ]:
# =============================================================================
# [셀 8] 3개 모델 Validation 성능 비교 안내
# =============================================================================

print("\n" + "=" * 60)
print("📊 다음 할 일")
print("=" * 60)
print("1. 위 학습 로그에서 best epoch의 Validation 성능 확인")
print("   → mAP50, Recall, Precision 기록하기")
print("")
print("2. 3개 모델 Validation 비교:")
print(f"   {'모델':<25} {'mAP50':>8} {'Recall':>8} {'Precision':>10}")
print(f"   {'-'*55}")
print(f"   {'YOLOv8n Default':<25} {'0.755':>8} {'0.696':>8} {'0.787':>10}")
print(f"   {'YOLOv8n Tuned(강한)':<25} {'0.747':>8} {'0.664':>8} {'0.746':>10}")
print(f"   {'YOLOv8n Light Tuned':<25} {'???':>8} {'???':>8} {'???':>10}")
print("")
print("3. Val 성능이 Default보다 좋으면 → Test 평가 진행")
print("   Val 성능이 Default보다 나쁘면 → Default 최종 확정")
print("=" * 60)
